# Customer Segmentation

## Point 1 — RFM Feature Table

In [ ]:
# بنستورد المكتبات اللي هنحتاجها في المشروع
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler
from sklearn.cluster import KMeans
from sklearn.metrics import silhouette_score
from sklearn.decomposition import PCA

# بنقرأ الداتا من ملف المشروع
# ولو الملف مش موجود في المسار ده، بنقرأه من GitHub
try:
    df = pd.read_csv("../../../../01-ml-project/retail_transactions_segmentation.csv")
except FileNotFoundError:
    df = pd.read_csv("https://raw.githubusercontent.com/HozaifaMoustafa/arabian-ai-training/main/01-ml-project/retail_transactions_segmentation.csv")

# بنحوّل عمود التاريخ لـ datetime عشان نقدر نتعامل مع التواريخ ونحسب الفرق بينهم
df["transaction_date"] = pd.to_datetime(df["transaction_date"])

# هنا بنحدد آخر تاريخ موجود في الداتا ونستخدمه كتاريخ مرجعي لحساب Recency
reference_date = df["transaction_date"].max()

# بنحسب الـ RFM لكل عميل:
# Recency = العميل اشترى آخر مرة من كام يوم
# Frequency = العميل عمل كام عملية شراء
# Monetary = العميل صرف إجمالي كام
rfm = df.groupby("customer_id").agg(
    Recency=("transaction_date", lambda x: (reference_date - x.max()).days),
    Frequency=("customer_id", "count"),
    Monetary=("amount", "sum")
).reset_index()

print("Reference date:", reference_date)
display(rfm.head())
print(rfm.shape)

Recency is measured from the maximum transaction date in the dataset. Monetary is the total spend for each customer.

## Point 2 — Features Scaled Before Clustering

The three RFM features have different scales, so they are standardized before K-Means.

In [ ]:
# بنحدد الأعمدة اللي هنستخدمها في تحليل RFM
features = ["Recency", "Frequency", "Monetary"]

# بنعمل Standardization عشان كل المتغيرات تبقى على مقياس قريب من بعض
# وده مهم لأن مقاييس Recency وFrequency وMonetary مختلفة
scaler = StandardScaler()
X = scaler.fit_transform(rfm[features])

## Point 3 — Choice of k

In [ ]:
# بنجرب أكتر من قيمة لـ k عشان نعرف أنهي عدد Clusters أنسب
k_values = range(2, 11)
inertia = []
silhouette = []

for k in k_values:
    # بنعمل نموذج K-Means بالقيمة الحالية لـ k
    model = KMeans(n_clusters=k, random_state=42, n_init=10)
    labels = model.fit_predict(X)

    # بنسجل قيمة Inertia عشان نستخدمها في Elbow Method
    inertia.append(model.inertia_)

    # بنحسب Silhouette Score عشان نشوف فصل الـ Clusters عامل إزاي
    silhouette.append(silhouette_score(X, labels))

In [ ]:
# بنرسم Elbow Method عشان نقارن الـ Inertia مع عدد الـ Clusters
plt.figure(figsize=(7, 4))
plt.plot(k_values, inertia, marker="o")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.title("Elbow Method")
plt.show()

# بنرسم Silhouette Score لكل قيمة من قيم k
# كل ما الـ Score يكون أعلى، بيكون فصل الـ Clusters أفضل
plt.figure(figsize=(7, 4))
plt.plot(k_values, silhouette, marker="o")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score")
plt.show()

The silhouette score is highest at **k = 2**. The elbow plot also shows that the biggest improvement happens early, so two clusters give a simple and well-separated segmentation. Therefore, I selected **k = 2**.


In [ ]:
# بنختار قيمة k اللي حققت أعلى Silhouette Score
# والقيمة دي هنستخدمها في نموذج K-Means النهائي
best_k = list(k_values)[int(np.argmax(silhouette))]
print("Selected k:", best_k)

## Point 4 — K-Means

In [ ]:
# بنطبق K-Means باستخدام قيمة k اللي اخترناها
# random_state=42 عشان النتيجة تفضل ثابتة لما نشغل الكود تاني
kmeans = KMeans(n_clusters=best_k, random_state=42, n_init=10)

# بنضيف رقم الـ Cluster لكل عميل
rfm["Cluster"] = kmeans.fit_predict(X)

display(rfm.head())

## Point 5 — PCA Visualization

In [ ]:
# بنستخدم PCA عشان نقلل أبعاد بيانات RFM من 3 متغيرات لمكونين بس
# الهدف إننا نقدر نشوف الـ Clusters في رسم ثنائي الأبعاد
pca = PCA(n_components=2)
X_pca = pca.fit_transform(X)

# بنرسم العملاء بعد استخدام PCA
# كل لون بيمثل Cluster مختلف
plt.figure(figsize=(8, 5))
sns.scatterplot(
    x=X_pca[:, 0],
    y=X_pca[:, 1],
    hue=rfm["Cluster"],
    palette="tab10"
)
plt.xlabel("PC1")
plt.ylabel("PC2")
plt.title("Customer Clusters")
plt.legend(title="Cluster")
plt.show()

## Point 6 — Customer Personas

In [ ]:
# بنحسب متوسط الـ RFM لكل Cluster
# المتوسطات دي بتساعدنا نفهم طبيعة العملاء في كل مجموعة
cluster_means = rfm.groupby("Cluster")[features].mean().round(2)
display(cluster_means)

**VIP:** These customers have the stronger RFM profile, with higher spending and more frequent purchases. They should be retained with loyalty rewards and personalized offers. The main goal is to keep their purchase frequency and spending high.

**At-Risk:** These customers have the weaker RFM profile, with lower spending and less frequent purchasing behavior. They should be targeted with reminders or promotions to encourage another purchase. The main goal is to increase their activity and move them toward a stronger customer segment.

In [ ]:
# بنشوف أنهي Cluster عنده أعلى قيمة Monetary
# وهنعتبره مجموعة الـ VIP
vip_cluster = cluster_means["Monetary"].idxmax()

# الـ Cluster التاني هنعامله كـ At-Risk
at_risk_cluster = [c for c in cluster_means.index if c != vip_cluster][0]

# بنسمي كل Cluster باسم واضح بدل الرقم بس
persona_names = {
    vip_cluster: "VIP",
    at_risk_cluster: "At-Risk"
}

# بنضيف اسم الـ Persona لكل عميل حسب الـ Cluster بتاعه
rfm["Persona"] = rfm["Cluster"].map(persona_names)

display(rfm.head())